# Lab 1: Environment Setup and Validation - Solution

This notebook contains the complete solution for Lab 1. Use this to verify your implementation or if you get stuck.

## Step 1: Verify Component Status

For Kubernetes Setup:
```bash
# Check all pods in iceberg namespace
kubectl get pods -n iceberg
```

For Docker Compose Setup:
```bash
# Check all containers
docker-compose ps
```

## Step 2: Test Storage Connectivity

For MinIO (Docker Compose):
```bash
mc alias set local http://localhost:9000 minioadmin minioadmin
mc ls local/
mc mb local/spark-logs
```

For ObjectScale (Kubernetes):
```bash
aws --endpoint-url=http://localhost:6080 s3 ls
aws --endpoint-url=http://localhost:6080 s3 mb s3://spark-logs
```

## Step 3: Test Polaris Catalog Connectivity

```bash
# Test health endpoint
curl -f http://localhost:8181/health
# Expected output: {"status":"healthy"}
```

## Step 4: Configure Spark for Iceberg

```bash
# Create spark configuration directory
mkdir -p ~/iceberg-labs/config

# Create spark-defaults.conf
cat > ~/iceberg-labs/config/spark-defaults.conf << 'EOF'
# Iceberg Configuration
spark.sql.catalog.iceberg=org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.iceberg.type=rest
spark.sql.catalog.iceberg.uri=http://localhost:8181/api/catalog

# S3 Configuration (for MinIO)
spark.hadoop.fs.s3a.impl=org.apache.hadoop.fs.s3a.S3AFileSystem
spark.hadoop.fs.s3a.path.style.access=true
spark.hadoop.fs.s3a.endpoint=http://localhost:9000
spark.hadoop.fs.s3a.access.key=minioadmin
spark.hadoop.fs.s3a.secret.key=minioadmin

# Event Logging
spark.eventLog.enabled=true
spark.eventLog.dir=s3a://spark-logs/
spark.history.fs.logDirectory=s3a://spark-logs/
EOF
```

## Step 5: Run First Iceberg Query

Start Spark shell with Iceberg configuration:
```bash
spark-shell \
  --packages org.apache.iceberg:iceberg-spark-runtime-3.5:1.5.0 \
  --conf spark.sql.catalog.iceberg=org.apache.iceberg.spark.SparkCatalog \
  --conf spark.sql.catalog.iceberg.type=rest \
  --conf spark.sql.catalog.iceberg.uri=http://localhost:8181/api/catalog \
  --conf spark.hadoop.fs.s3a.endpoint=http://localhost:9000 \
  --conf spark.hadoop.fs.s3a.access.key=minioadmin \
  --conf spark.hadoop.fs.s3a.secret.key=minioadmin \
  --conf spark.hadoop.fs.s3a.path.style.access=true
```

Once in the Spark shell, run:

In [ ]:
// Create a simple Iceberg table
spark.sql("""
  CREATE TABLE iceberg.default.test_table (
    id INT,
    name STRING,
    value DOUBLE
  ) USING iceberg
  PARTITIONED BY (name)
  TBLPROPERTIES (
    'format-version'='2'
  )
""")

In [ ]:
// Insert test data
spark.sql("""
  INSERT INTO iceberg.default.test_table VALUES
    (1, 'Alice', 100.0),
    (2, 'Bob', 200.0),
    (3, 'Charlie', 300.0)
""")

In [ ]:
// Query the table
val result = spark.sql("SELECT * FROM iceberg.default.test_table")
result.show()

In [ ]:
// Assertion: Verify table exists and has correct data
val tableExists = spark.catalog.tableExists("iceberg.default.test_table")
assert(tableExists, "Table should exist")

val count = spark.sql("SELECT COUNT(*) FROM iceberg.default.test_table").collect()(0).getLong(0)
assert(count == 3, s"Table should have 3 rows, but has $count")

println("✅ All assertions passed!")

## Step 6: Verify Spark History Server

```bash
# Access Spark History Server UI
# Open browser to: http://localhost:18080

# Or test via curl
curl -f http://localhost:18080
# Expected: HTML response from History Server UI
```

## ✅ Lab Completion Checklist

- [x] All components running (Kubernetes pods or Docker containers)
- [x] Storage accessible and spark-logs bucket created
- [x] Polaris catalog health check successful
- [x] Spark configuration file created
- [x] First Iceberg table created successfully
- [x] Data inserted and queried successfully
- [x] Spark History Server accessible

## 🎓 Key Concepts Learned

1. **Apache Polaris**: REST-based Iceberg catalog service
2. **S3-Compatible Storage**: MinIO and ObjectScale as alternatives to AWS S3
3. **Spark History Server**: UI for viewing completed Spark jobs
4. **Iceberg Tables**: Apache Iceberg table format and operations
5. **Configuration Management**: Spark configuration for Iceberg operations